# Find the Promos — Colab runner
I upload the upgraded project ZIP, choose Gemini or explicit MOCK mode, then run the full input.
For Gemini, add the secret `GEMINI_API_KEY` in Colab's key panel and grant notebook access.
Keep this notebook and its data private. Mock results are fixtures, not recommendations.


In [ ]:
# 1. Install optional YAML support and upload the project ZIP.
import sys, subprocess, zipfile, stat, os
from pathlib import Path
from google.colab import files
subprocess.run([sys.executable, '-m', 'pip', 'install', 'PyYAML>=6.0,<7.0'], check=True)
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one upgraded project ZIP.'
dest = Path('/content/kuku_project')
dest.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    assert sum(i.file_size for i in z.infolist()) < 300 * 1024 * 1024
    for member in z.infolist():
        p = Path(member.filename)
        assert not p.is_absolute() and not p.drive and '..' not in p.parts and '\\' not in member.filename
        assert not stat.S_ISLNK(member.external_attr >> 16)
    z.extractall(dest)
roots = [p.parent for p in dest.rglob('llm_layer.py')]
assert len(roots) == 1, 'Expected one project containing llm_layer.py.'
os.chdir(roots[0])
print('Project ready:', Path.cwd().name)


In [ ]:
# 2. Explicit model mode; secrets are read but never printed or saved.
import json
from google.colab import userdata
MODE = 'gemini'  # Set to 'mock' to test offline.
assert MODE in ('gemini', 'mock')
cfg = json.loads(Path('config.yaml').read_text())
cfg['provider'] = MODE
if MODE == 'gemini':
    try:
        key = userdata.get('GEMINI_API_KEY')
        assert key
    except Exception:
        raise RuntimeError('Add GEMINI_API_KEY to Colab secrets and grant access, or explicitly set MODE to mock.') from None
    os.environ['GEMINI_API_KEY'] = key
    del key
# Change cfg['model'] here if desired, and update cfg['pricing'].
Path('config.colab.json').write_text(json.dumps(cfg, indent=2))
print('Mode:', MODE, '| Model:', cfg['model'])


In [ ]:
# 3. Choose the bundled pack or upload a new screenplay ZIP.
USE_BUNDLED_INPUT = False  # Upload your screenplay ZIP in this cell (cell 3).
input_path = 'input'
if not USE_BUNDLED_INPUT:
    new_upload = files.upload()
    zips = [name for name in new_upload if name.lower().endswith('.zip')]
    assert len(zips) == 1, 'Upload one screenplay ZIP, not another project ZIP.'
    input_path = zips[0]


In [ ]:
# 4. Run parser, wide candidates and A/B/C. A quota failure leaves raw diagnostics/cache.
subprocess.run([sys.executable, 'run.py', '--input', input_path,
                '--out', 'outputs_colab', '--llm', '--config', 'config.colab.json',
                '--llm-out', 'outputs_llm_colab'], check=True)
# Optional corpus regression tests (always use the supplied input for these tests):
# subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


In [ ]:
# 5. Remove overlapping promos and force the top 3 into different episodes (reads, never edits, promos_ranked.json)
subprocess.run([sys.executable, 'postprocess_diverse.py', 'outputs_llm_colab'], check=True)


In [ ]:
# 6. Inspect and download the complete run, including raw responses and replay cache.
result = json.loads(Path('outputs_llm_colab/promos_ranked.json').read_text())
print(result['notice'])
print('Ranked:', len(result['promos']), '| Top three IDs:', result['top_3'])
import pandas as pd
from IPython.display import display
display(pd.read_csv('outputs_llm_colab/promos_ranked.csv').head(10))
with zipfile.ZipFile('colab_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['outputs_colab', 'outputs_llm_colab', '.cache/llm']:
        for p in sorted(Path(folder).rglob('*')):
            if p.is_file(): z.write(p, p.as_posix())
    z.write('config.colab.json')  # Contains environment variable NAME, never the secret.
files.download('colab_results.zip')
